In [1]:
with open("output.txt" ) as f:
      data = f.read()
print(len(data))

4749


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500
chunk_overlap = 75

splitter = RecursiveCharacterTextSplitter(
    chunk_size = chunk_size,
    chunk_overlap= chunk_overlap
)

chunks = splitter.split_text(data)

In [19]:
len(chunks)

12

In [20]:
docs = chunks

In [21]:
from sentence_transformers import SentenceTransformer

emd_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')





NameError: name 'emd_data' is not defined

In [22]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

In [23]:
from qdrant_client.models import VectorParams, Distance

client.create_collection(
    collection_name = "documents",
    vectors_config = VectorParams(
        size = 384,
        distance = Distance.COSINE
    )
)

True

In [24]:
from qdrant_client.models import PointStruct

emd_docs = emd_model.encode(docs)

points = [
    PointStruct(
        id=i,
        vector=emd_docs[i],
        payload={"text": docs[i]}
    )
    for i in range(len(docs))
]

client.upsert(
    collection_name="documents",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
query = "what is Embeddings (Short & Practical) "

query_vector = emd_model.encode_query(query)

results = client.query_points(
    collection_name="documents",
    query=query_vector,
    limit=4
)

In [26]:
results

QueryResponse(points=[ScoredPoint(id=3, version=0, score=0.5117073396252394, payload={'text': 'When to use: LLMs (GPT-style), encoders (BERT), encoder-decoder (T5).\n2. Embeddings (Short & Practical) 📏\nDefinition: A  dense  vector  representing  a  word/sentence/document  so  semantic  similarity  ≈ vector\nsimilarity.\nProgression: Word2Vec  → Contextual  embeddings  (BERT)  → Sentence  embeddings  (sentence-\ntransformers).\n1\n\x0cQuick example — compute sentence embeddings & similarity (Python):\n# pip install -U sentence-transformers\nfrom sentence_transformers import SentenceTransformer'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=9, version=0, score=0.4962569520025562, payload={'text': 'Embedding dim (d): Size of token vectors.\nLatent diffusion: Diffusion in compressed latent space (faster, used by Stable Diffusion).\nPrompt: Text instructions for conditional generation.\n6. Exercises (tiny, 10–20 min each) ✅\nTransformer sketch: Draw blocks for embedding 

In [27]:
results.points[0].id

3

In [28]:
for i in range(4):
    print("**************************************")
    print(f' chunk id : {results.points[i].id}')
    print(f'chunk text : {results.points[i].payload["text"]}')
    print("**************************************")


**************************************
 chunk id : 3
chunk text : When to use: LLMs (GPT-style), encoders (BERT), encoder-decoder (T5).
2. Embeddings (Short & Practical) 📏
Definition: A  dense  vector  representing  a  word/sentence/document  so  semantic  similarity  ≈ vector
similarity.
Progression: Word2Vec  → Contextual  embeddings  (BERT)  → Sentence  embeddings  (sentence-
transformers).
1
Quick example — compute sentence embeddings & similarity (Python):
# pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer
**************************************
**************************************
 chunk id : 9
chunk text : Embedding dim (d): Size of token vectors.
Latent diffusion: Diffusion in compressed latent space (faster, used by Stable Diffusion).
Prompt: Text instructions for conditional generation.
6. Exercises (tiny, 10–20 min each) ✅
Transformer sketch: Draw blocks for embedding → attention → FFN for a 2-layer transformer on a
piece of paper.